In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Read raw files as text via Auto Loader
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/orders_schema")
    .load("/Volumes/streaming_project/ecommerce/stream_volume/orders")
)

# Parse JSON array from text
orders_schema = ArrayType(StructType([
    StructField("id", LongType()),
    StructField("userId", LongType()),
    StructField("date", StringType()),
    StructField("products", ArrayType(StructType([
        StructField("productId", LongType()),
        StructField("quantity", LongType())
    ])))
]))

orders_stream = (
    raw_stream
    .select(explode(from_json(col("value"), orders_schema)).alias("data"))
    .select(
        col("data.id").alias("order_id"),
        col("data.userId").alias("customer_id"),
        col("data.date").alias("order_date"),
        explode(col("data.products")).alias("product")
    )
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_date"),
        col("product.productId").alias("product_id"),
        col("product.quantity").alias("quantity")
    )
)

(
    orders_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/bronze_orders")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.bronze_orders")
)

In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Read raw files as text via Auto Loader
raw_customers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/customers_schema")
    .load("/Volumes/streaming_project/ecommerce/stream_volume/customers")
)

customers_schema = ArrayType(StructType([
    StructField("id", LongType()),
    StructField("email", StringType()),
    StructField("username", StringType()),
    StructField("password", StringType()),
    StructField("phone", StringType()),
    StructField("name", StructType([
        StructField("firstname", StringType()),
        StructField("lastname", StringType())
    ])),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("street", StringType()),
        StructField("number", LongType()),
        StructField("zipcode", StringType()),
        StructField("geolocation", StructType([
            StructField("lat", StringType()),
            StructField("long", StringType())
        ]))
    ]))
]))

customers_stream = (
    raw_customers_stream
    .select(explode(from_json(col("value"), customers_schema)).alias("data"))
    .select(
        col("data.id").alias("customer_id"),
        col("data.username").alias("username"),
        col("data.email").alias("email"),
        col("data.phone").alias("phone"),
        col("data.name.firstname").alias("first_name"),
        col("data.name.lastname").alias("last_name"),
        col("data.address.city").alias("city"),
        col("data.address.street").alias("street"),
        col("data.address.number").alias("street_number"),
        col("data.address.zipcode").alias("zipcode"),
        col("data.address.geolocation.lat").alias("lat"),
        col("data.address.geolocation.long").alias("long")
    )
)

(
    customers_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/bronze_customers")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.bronze_customers")
)

print("✅ bronze_customers stream started")



In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Read raw files as text via Auto Loader
raw_products_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/products_schema")
    .load("/Volumes/streaming_project/ecommerce/stream_volume/products")
)

products_schema = ArrayType(StructType([
    StructField("id", LongType()),
    StructField("title", StringType()),
    StructField("price", DoubleType()),
    StructField("description", StringType()),
    StructField("category", StringType()),
    StructField("image", StringType()),
    StructField("rating", StructType([
        StructField("rate", DoubleType()),
        StructField("count", LongType())
    ]))
]))

products_stream = (
    raw_products_stream
    .select(explode(from_json(col("value"), products_schema)).alias("data"))
    .select(
        col("data.id").alias("product_id"),
        col("data.title").alias("title"),
        col("data.price").alias("price"),
        col("data.description").alias("description"),
        col("data.category").alias("category"),
        col("data.image").alias("image"),
        col("data.rating.rate").alias("rating_rate"),
        col("data.rating.count").alias("rating_count")
    )
)

(
    products_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/bronze_products")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.bronze_products")
)

print("✅ bronze_products stream started")

In [0]:
display(spark.read.table("streaming_project.ecommerce.bronze_products"))


product_id,title,price,description,category,image,rating_rate,rating_count
1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",109.95,"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png,3.9,120
2,Mens Casual Premium Slim Fit T-Shirts,22.3,"Slim-fitting style, contrast raglan long sleeve, three-button henley placket, light weight & soft fabric for breathable and comfortable wearing. And Solid stitched shirts with round neck made for durability and a great fit for casual fashion wear and diehard baseball fans. The Henley style round neckline includes a three-button placket.",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_SY879._SX._UX._SY._UY_t.png,4.1,259
3,Mens Cotton Jacket,55.99,"great outerwear jackets for Spring/Autumn/Winter, suitable for many occasions, such as working, hiking, camping, mountain/rock climbing, cycling, traveling or other outdoors. Good gift choice for you or your family member. A warm hearted love to Father, husband or son in this thanksgiving or Christmas Day.",men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_UX679_t.png,4.7,500
4,Mens Casual Slim Fit,15.99,"The color could be slightly different between on the screen and in practice. / Please note that body builds vary by person, therefore, detailed size information should be reviewed below on the product description.",men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_UY879_t.png,2.1,430
5,John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet,695.0,"From our Legends Collection, the Naga was inspired by the mythical water dragon that protects the ocean's pearl. Wear facing inward to be bestowed with love and abundance, or outward for protection.",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_UL640_QL65_ML3_t.png,4.6,400
6,Solid Gold Petite Micropave,168.0,Satisfaction Guaranteed. Return or exchange any order within 30 days.Designed and sold by Hafeez Center in the United States. Satisfaction Guaranteed. Return or exchange any order within 30 days.,jewelery,https://fakestoreapi.com/img/61sbMiUnoGL._AC_UL640_QL65_ML3_t.png,3.9,70
7,White Gold Plated Princess,9.99,"Classic Created Wedding Engagement Solitaire Diamond Promise Ring for Her. Gifts to spoil your love more for Engagement, Wedding, Anniversary, Valentine's Day...",jewelery,https://fakestoreapi.com/img/71YAIFU48IL._AC_UL640_QL65_ML3_t.png,3.0,400
8,Pierced Owl Rose Gold Plated Stainless Steel Double,10.99,Rose Gold Plated Double Flared Tunnel Plug Earrings. Made of 316L Stainless Steel,jewelery,https://fakestoreapi.com/img/51UDEzMJVpL._AC_UL640_QL65_ML3_t.png,1.9,100
9,WD 2TB Elements Portable External Hard Drive - USB 3.0,64.0,"USB 3.0 and USB 2.0 Compatibility Fast data transfers Improve PC Performance High Capacity; Compatibility Formatted NTFS for Windows 10, Windows 8.1, Windows 7; Reformatting may be required for other operating systems; Compatibility may vary depending on user’s hardware configuration and operating system",electronics,https://fakestoreapi.com/img/61IBBVJvSDL._AC_SY879_t.png,3.3,203
10,SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s,109.0,"Easy upgrade for faster boot up, shutdown, application load and response (As compared to 5400 RPM SATA 2.5” hard drive; Based on published specifications and internal benchmarking tests using PCMark vantage scores) Boosts burst write performance, making it ideal for typical PC workloads The perfect balance of performance and reliability Read/write speeds of up to 535MB/s/450MB/s (Based on internal testing; Performance may vary depending upon drive capacity, host device, OS and application.)",electronics,https://fakestoreapi.com/img/61U7T1koQqL._AC_SX679_t.png,2.9,470
